# PM10 1시간 뒤 등급 예측 모델 학습 (2023-2024)

## 0. 패키지 설치

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn tensorflow openpyxl xlrd xgboost
print('✅ 라이브러리 설치 완료')

## 1. 환경 설정

In [ ]:
import os

dirs = [
    "data/raw",
    "data/merged",
    "data/featured",
    "data/model_input",
    "models/preprocess",
    "models/tensorflow",
    "models/tree",
    "outputs/plots",
    "outputs/predictions",
    "outputs/metrics"
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print('✅ 디렉토리 생성 완료')

## 2. 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import joblib

print('✅ 라이브러리 로드 완료')

## 3. 데이터 불러오기

In [ ]:
dust_2023 = pd.read_csv('dust_2023.csv')
dust_2024 = pd.read_csv('dust_2024.csv')
weather_2023 = pd.read_csv('weather_2023.csv')
weather_2024 = pd.read_csv('weather_2024.csv')

print('dust_2023:', dust_2023.shape)
print('dust_2024:', dust_2024.shape)
print('weather_2023:', weather_2023.shape)
print('weather_2024:', weather_2024.shape)

## 4. 데이터 전처리 및 병합

In [ ]:
def preprocess_merge(dust, weather):
    dust = dust.copy()
    weather = weather.copy()

    dust.columns = dust.columns.str.strip()
    weather.columns = weather.columns.str.strip()

    dust['측정일시'] = pd.to_datetime(
        dust['측정일시'].astype(str).str.strip(),
        format='%Y%m%d%H',
        errors='coerce'
    )
    weather['일시'] = pd.to_datetime(
        weather['일시'].astype(str).str.strip(),
        errors='coerce'
    )

    dust = dust.rename(columns={'측정일시': 'datetime'})
    weather = weather.rename(columns={'일시': 'datetime'})

    dust = dust.dropna(subset=['datetime'])
    weather = weather.dropna(subset=['datetime'])

    df = pd.merge(dust, weather, on='datetime', how='inner')
    df = df.sort_values(['datetime', '지역', '측정소코드']).reset_index(drop=True)

    return df


# QC 플래그 컬럼 제거 함수
def drop_qc_cols(df):
    qc_cols = [c for c in df.columns if 'QC' in c]
    print('QC 제거:', qc_cols)
    return df.drop(columns=qc_cols)


merged_2023 = preprocess_merge(dust_2023, weather_2023)
merged_2024 = preprocess_merge(dust_2024, weather_2024)

merged_2023 = drop_qc_cols(merged_2023)
merged_2024 = drop_qc_cols(merged_2024)

# 서울 필터링
train_df = merged_2023[merged_2023['지점명'] == '서울'].copy().reset_index(drop=True)
test_df  = merged_2024[merged_2024['지점명'] == '서울'].copy().reset_index(drop=True)

print('서울 필터링 후 train_df:', train_df.shape)
print('서울 필터링 후 test_df :', test_df.shape)

# 데이터 샘플 확인
display(train_df.head())

In [ ]:
train_df.to_csv('data/merged/train_2023_merged.csv', index=False, encoding='utf-8-sig')
test_df.to_csv('data/merged/test_2024_merged.csv',   index=False, encoding='utf-8-sig')
print('✅ merged 저장 완료')

## 5. 피처 엔지니어링

In [ ]:
# PM10 4단계 등급 레이블
def pm10_grade(pm10):
    if pm10 <= 30:  return 0  # 좋음
    elif pm10 <= 80:  return 1  # 보통
    elif pm10 <= 150: return 2  # 나쁨
    else:             return 3  # 매우나쁨

for df in [train_df, test_df]:
    df.sort_values(['측정소코드', 'datetime'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df['PM10_future_1h'] = df.groupby('측정소코드')['PM10'].shift(-1)
    df['grade_future_1h'] = df['PM10_future_1h'].apply(pm10_grade)

train_df.dropna(subset=['PM10_future_1h'], inplace=True)
test_df.dropna(subset=['PM10_future_1h'],  inplace=True)
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print('클래스 분포 (train):')
print(train_df['grade_future_1h'].value_counts().sort_index())
print('\n클래스 분포 (test):')
print(test_df['grade_future_1h'].value_counts().sort_index())

In [ ]:
# 시간 파생변수
def add_time_features(df):
    df = df.copy()
    df['month']     = df['datetime'].dt.month
    df['day']       = df['datetime'].dt.day
    df['hour']      = df['datetime'].dt.hour
    df['dayofweek'] = df['datetime'].dt.dayofweek

    def season_map(m):
        if m in [3,4,5]:   return 'Spring'
        elif m in [6,7,8]: return 'Summer'
        elif m in [9,10,11]: return 'Autumn'
        else:              return 'Winter'

    df['season'] = df['month'].apply(season_map)
    return df

train_df = add_time_features(train_df)
test_df  = add_time_features(test_df)

In [ ]:
# 강수 파생변수
def add_precip_features(df):
    df = df.copy()
    if '강수량(mm)' in df.columns:
        df['precip_class']    = df['강수량(mm)'].apply(
            lambda x: np.nan if pd.isna(x) else (0 if x==0 else (1 if x<3 else (2 if x<15 else 3)))
        )
        df['precip_weighted'] = df['강수량(mm)'].apply(
            lambda x: x*2.0 if pd.notna(x) and x>=15 else x
        )
        df['rain_binary']     = df['강수량(mm)'].apply(
            lambda x: 1 if pd.notna(x) and x>0 else 0
        )
    return df

train_df = add_precip_features(train_df)
test_df  = add_precip_features(test_df)

In [ ]:
# lag 피처
def add_lag_features(df):
    df = df.copy()
    df = df.sort_values(['측정소코드', 'datetime']).reset_index(drop=True)
    df['pm10_lag1']   = df.groupby('측정소코드')['PM10'].shift(1)
    df['pm10_lag2']   = df.groupby('측정소코드')['PM10'].shift(2)
    df['pm10_lag3']   = df.groupby('측정소코드')['PM10'].shift(3)
    df['pm25_lag1']   = df.groupby('측정소코드')['PM25'].shift(1)
    df['pm25_lag2']   = df.groupby('측정소코드')['PM25'].shift(2)
    df['precip_lag1'] = df.groupby('측정소코드')['강수량(mm)'].shift(1)
    df['precip_lag2'] = df.groupby('측정소코드')['강수량(mm)'].shift(2)
    return df

train_df = add_lag_features(train_df)
test_df  = add_lag_features(test_df)

In [ ]:
train_df.to_csv('data/featured/train_2023_featured.csv', index=False, encoding='utf-8-sig')
test_df.to_csv('data/featured/test_2024_featured.csv',   index=False, encoding='utf-8-sig')
print('✅ featured 저장 완료')

## 6. 모델 입력 데이터 구성

In [ ]:
exclude_cols = [
    'grade_future_1h', 'PM10_future_1h', 'datetime', 'PM10',
    '주소', '측정소명', '지점명', '강수량(mm)'
]

candidate_cols = [c for c in train_df.columns if c not in exclude_cols]
print('입력 후보 컬럼 수:', len(candidate_cols))

In [ ]:
# 수치형 / 범주형 분류
numeric_cols     = []
categorical_cols = []

for col in candidate_cols:
    series = train_df[col]
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_categorical_dtype(series):
        categorical_cols.append(col)
    elif pd.api.types.is_numeric_dtype(series):
        nunique      = series.nunique(dropna=True)
        unique_ratio = nunique / max(len(series.dropna()), 1)
        if nunique <= 20 and unique_ratio < 0.01:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)

force_categorical = ['month', 'season', '측정소코드', '지역', 'precip_class', 'rain_binary']
for col in force_categorical:
    if col in numeric_cols:
        numeric_cols.remove(col)
    if col not in categorical_cols:
        categorical_cols.append(col)

categorical_cols = list(dict.fromkeys(categorical_cols))

print('numeric_cols:', len(numeric_cols))
print('categorical_cols:', len(categorical_cols))

In [ ]:
needed_cols = numeric_cols + categorical_cols + ['grade_future_1h', 'datetime']

train_model_df = train_df[needed_cols].copy()
test_model_df  = test_df[needed_cols].copy()

print('train_model_df:', train_model_df.shape)
print('test_model_df :', test_model_df.shape)
display(train_model_df.head())

In [ ]:
# 결측 처리
numeric_cols     = [c for c in numeric_cols     if c in train_model_df.columns]
categorical_cols = [c for c in categorical_cols if c in train_model_df.columns]

all_nan_numeric     = [c for c in numeric_cols     if train_model_df[c].isnull().all()]
all_nan_categorical = [c for c in categorical_cols if train_model_df[c].isnull().all()]

numeric_cols     = [c for c in numeric_cols     if c not in all_nan_numeric]
categorical_cols = [c for c in categorical_cols if c not in all_nan_categorical]

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

if numeric_cols:
    train_model_df[numeric_cols] = num_imputer.fit_transform(train_model_df[numeric_cols])
    test_model_df[numeric_cols]  = num_imputer.transform(test_model_df[numeric_cols])

if categorical_cols:
    train_model_df[categorical_cols] = cat_imputer.fit_transform(train_model_df[categorical_cols])
    test_model_df[categorical_cols]  = cat_imputer.transform(test_model_df[categorical_cols])

print('결측 처리 후 train missing:', train_model_df.isnull().sum().sum())
print('결측 처리 후 test  missing:', test_model_df.isnull().sum().sum())

In [ ]:
# 원-핫 인코딩
train_encoded = pd.get_dummies(train_model_df, columns=categorical_cols, drop_first=False)
test_encoded  = pd.get_dummies(test_model_df,  columns=categorical_cols, drop_first=False)

train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

print('train_encoded:', train_encoded.shape)
print('test_encoded :', test_encoded.shape)

In [ ]:
train_encoded.to_csv('data/model_input/train_2023_model_input.csv', index=False, encoding='utf-8-sig')
test_encoded.to_csv('data/model_input/test_2024_model_input.csv',   index=False, encoding='utf-8-sig')
print('✅ 모델 입력 데이터 저장 완료')

## 7. 탐색적 분석 (EDA)

In [ ]:
# target과의 상관관계 상위 20개
all_features = [c for c in train_encoded.columns if c not in ['grade_future_1h', 'datetime']]

# corrwith()로 target과의 상관만 계산 (전체 corr()보다 훨씬 빠름)
corr_with_target = train_encoded[all_features].corrwith(train_encoded['grade_future_1h'])
top20_corr = corr_with_target.abs().sort_values(ascending=False).head(20)

plt.figure(figsize=(8, 6))
top20_corr.sort_values().plot(kind='barh')
plt.title('Target(grade_future_1h)과 상관관계 상위 20 피처')
plt.tight_layout()
plt.savefig('outputs/plots/plot_target_corr_top20.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. 1차 모델 학습 (피처 선택용)

In [ ]:
X_train = train_encoded[all_features]
y_train = train_encoded['grade_future_1h'].astype(int)
X_test  = test_encoded[all_features]
y_test  = test_encoded['grade_future_1h'].astype(int)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

In [ ]:
# 1차 XGBoost (feature importance 추출용)
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train_scaled, y_train)
print('✅ 1차 XGBoost 학습 완료')

In [ ]:
# Feature Importance Top 20 시각화
importance_df = pd.DataFrame({
    'feature':    X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(12, 10))
importance_df.head(20).sort_values('importance').plot(
    x='feature', y='importance', kind='barh', legend=False
)
plt.title('XGBoost Feature Importance Top 20')
plt.tight_layout()
plt.savefig('outputs/plots/plot_xgb_importance_top20.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 중요도 기준 피처 선택
threshold = 0.001

important_features = importance_df[
    importance_df['importance'] > threshold
]['feature'].tolist()

print('선택된 feature 수:', len(important_features))
print(important_features[:20])

## 9. 최종 모델 학습

In [ ]:
# 선택된 피처로 재구성
X_train_final = X_train[important_features]
X_test_final  = X_test[important_features]

print('X_train_final:', X_train_final.shape)
print('X_test_final :', X_test_final.shape)

In [ ]:
# 스케일링
scaler_final = StandardScaler()
X_train_final_scaled = scaler_final.fit_transform(X_train_final)
X_test_final_scaled  = scaler_final.transform(X_test_final)

In [ ]:
# 클래스 가중치
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict   = {cls: w for cls, w in zip(classes, weights)}
sample_weight_final = pd.Series(y_train).map(class_weight_dict).values

print('class_weight_dict:', class_weight_dict)

In [ ]:
# TensorFlow
model_final = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_final_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(4, activation='softmax')
])

model_final.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_final.fit(
    X_train_final_scaled, y_train,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
# XGBoost (최종)
xgb_final = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_final.fit(X_train_final_scaled, y_train, sample_weight=sample_weight_final)

# RandomForest (최종)
rf_final = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_final.fit(X_train_final_scaled, y_train)

print('✅ 최종 모델 학습 완료')

## 10. 매우나쁨 Threshold 실험

In [ ]:
# XGBoost 확률 예측
xgb_probs = xgb_final.predict_proba(X_test_final_scaled)

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30]
results = []

for t in thresholds:
    preds = np.where(xgb_probs[:, 3] >= t, 3, xgb_probs.argmax(axis=1))
    acc     = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average='macro')
    recall_vb = f1_score(y_test, preds, average=None)[3]  # 매우나쁨 recall
    prec_vb   = classification_report(y_test, preds, output_dict=True)['3']['precision']
    results.append({
        'threshold':         t,
        'accuracy':          round(acc, 4),
        'macro_f1':          round(macro_f1, 4),
        'vb_precision':      round(prec_vb, 4),
        'vb_f1':             round(recall_vb, 4),
    })

threshold_df = pd.DataFrame(results)
display(threshold_df)

# 시각화
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df['threshold'], threshold_df['macro_f1'],  marker='o', label='Macro F1')
ax.plot(threshold_df['threshold'], threshold_df['vb_precision'], marker='s', label='매우나쁨 Precision')
ax.plot(threshold_df['threshold'], threshold_df['vb_f1'],     marker='^', label='매우나쁨 F1')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('매우나쁨 Threshold별 성능 변화')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/plots/plot_threshold_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n→ Macro F1과 매우나쁨 F1을 동시에 고려하여 최적 threshold를 선택하세요.')

## 11. 최종 평가

In [ ]:
# threshold 위 실험 결과 보고 선택
BEST_THRESHOLD = 0.20

tf_probs_final  = model_final.predict(X_test_final_scaled)
tf_pred_final   = np.where(tf_probs_final[:, 3] > 0.30, 3, tf_probs_final.argmax(axis=1))

xgb_pred_final  = np.where(xgb_probs[:, 3] >= BEST_THRESHOLD, 3, xgb_probs.argmax(axis=1))

rf_pred_final   = rf_final.predict(X_test_final_scaled)

for name, preds in [('TensorFlow', tf_pred_final),
                     ('XGBoost',    xgb_pred_final),
                     ('RandomForest', rf_pred_final)]:
    print(f'=== FINAL {name} ===')
    print('Accuracy :', accuracy_score(y_test, preds))
    print('Macro F1 :', f1_score(y_test, preds, average='macro'))
    print(classification_report(y_test, preds, digits=4))
    print(confusion_matrix(y_test, preds))
    print()

## 12. 결과 저장

In [ ]:
final_result_df = pd.DataFrame({
    'datetime':             test_encoded['datetime'].values,
    'actual_grade_future_1h': y_test.values,
    'tf_pred_future_1h':    tf_pred_final,
    'xgb_pred_future_1h':   xgb_pred_final,
    'rf_pred_future_1h':    rf_pred_final
})

final_result_df.to_csv(
    'outputs/predictions/pred_2024_final.csv',
    index=False, encoding='utf-8-sig'
)
display(final_result_df.head())
print('✅ 예측 결과 저장 완료')

In [ ]:
# 모델 저장
model_final.save('models/tensorflow/dust_tf_model_final_v2.keras')
joblib.dump(xgb_final, 'models/tree/dust_xgb_model_final_v2.pkl')
joblib.dump(rf_final,  'models/tree/dust_rf_model_final_v2.pkl')

# 전처리 저장
joblib.dump(scaler_final,       'models/preprocess/dust_scaler_final_v2.pkl')
joblib.dump(important_features, 'models/preprocess/dust_selected_features_final.pkl')
joblib.dump(train_encoded.columns.tolist(), 'models/preprocess/dust_encoded_columns_final.pkl')

print('✅ 모델 및 전처리 파일 저장 완료')